# HF-Agent with Genie Worksheets (MVP)

This notebook using Genie Worksheets to achieve a minimal dialogue loop for heart failure medication follow-up: Intake → RiskScreen → Recommendation → Communicate (optional Physician Approval).

Preparation before running:

- Copy the "Starter Worksheet Template", create HF Worksheets according to the instructions below, and obtain the gsheet_id.

- Genie-worksheets is correctly installed and `.env` is configured (OpenAI/Azure, etc.).



## Worksheets Design
Create 4 tables (column names are case-sensitive):

1) HF_Intake

- Fields: `patient_name (str)`, `vitals_sbp (int)`, `vitals_dbp (int)`, `vitals_hr (int)`, `vitals_weight_kg (float)`, `labs_creatinine_mg_dl (float)`, `labs_egfr (int)`, `labs_potassium_mmol_l (float)`, `symptoms (list[str])`, `adherence (str)`

- Logic: Collect missing slots via AskField

2) HF_RiskScreen

- Call API: `evaluate_risk`, input the above fields; output `risk_level (str)`, `flags (list[str])`

- If high → jump to HF_Escalation; otherwise → HF_Recommendation

3) HF_Recommendation

- Calls the API: `create_recommendation`, inputting patient_state + strategy (optional);

- Outputs `rec_actions (str/json)`, `rec_monitoring (str/json)`, `rec_followup_weeks (int)`, `rec_tags (list[str])`

4) HF_Communicate

- Converts the results of the previous step into a natural language summary of the patient

Optional 5) HF_Physician_Approval (Asynchronous):

- Calls `request_approval` → `approval_status (pending/approved/denied)`; the round ends when pending, and continues via external events.


# Step1 Prepare the repository and install packages

In [ ]:
# Clone the repository

!git clone https://github.com/stanford-oval/genie-worksheets.git

fatal: destination path 'genie-worksheets' already exists and is not an empty directory.


In [ ]:
!cd genie-worksheets/; git checkout no-chainlite

Already on 'no-chainlite'
Your branch is up to date with 'origin/no-chainlite'.


In [ ]:
# install the required packages
!cd genie-worksheets; uv pip install -e .

Using Python 3.12.12 environment at: /usr
Resolved 194 packages in 1.66s
Prepared 1 package in 1.33s
Uninstalled 1 package in 5ms
Installed 1 package in 1ms
 ~ genie-worksheets==1.0.0b3 (from file:///content/genie-worksheets)


In [ ]:
import os
import sys
sys.path.append("/content/genie-worksheets/src/")
from worksheets.specification.from_spreadsheet import gsheet_to_classes

In [ ]:
env_content = """
LLM_API_KEY=sk-xG3QmIRkMDdOLG1lZ-ggHg
LLM_API_BASE_URL="https://cs224v-litellm.genie.stanford.edu"
"""

with open("/content/genie-worksheets/.env", "w") as f:
  f.write(env_content)

## Litellm Ping Test

In [ ]:
import requests
import json
from dotenv import load_dotenv

load_dotenv("/content/genie-worksheets/.env")

API_KEY = os.getenv("LLM_API_KEY")
BASE_URL = os.getenv("LLM_API_BASE_URL")

response = requests.post(
    f"{BASE_URL}/chat/completions",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "gpt-4.1",
        "messages": [
            {"role": "user", "content": "Say hi in one word"}
        ],
        "max_tokens": 5
    }
)

if response.status_code == 200:
    result = response.json()
    print("Success!")
    print(f"Response: {result['choices'][0]['message']['content']}")
    print(f"Tokens used: {result.get('usage', {}).get('total_tokens', 'N/A')}")
else:
    print(f"Error {response.status_code}: {response.text}")

Success!
Response: Hello!
Tokens used: 15


In [ ]:
current_dir = os.getcwd()
root_dir = os.path.join(current_dir, "genie-worksheets")

# Credentials for reading the spreadsheet
!curl -L -o creds.zip "https://drive.google.com/uc?export=download&id=11QvSs2JZ5qpPrCvbX66Dg8pxfM_B8GaH"
!unzip creds.zip -d genie-worksheets/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  3009  100  3009    0     0   1669      0  0:00:01  0:00:01 --:--:--  2167
Archive:  creds.zip
  inflating: genie-worksheets/worksheets/token.json  
  inflating: genie-worksheets/worksheets/credentials.json  
  inflating: genie-worksheets/worksheets/service_account.json  


In [ ]:

!mv "/content/genie-worksheets/worksheets/credentials.json" "/content/genie-worksheets/src/worksheets/config/credentials.json"
!mv "/content/genie-worksheets/worksheets/service_account.json" "/content/genie-worksheets/src/worksheets/config/service_account.json"
!mv "/content/genie-worksheets/worksheets/token.json" "/content/genie-worksheets/src/worksheets/config/token.json"

creds1 = os.path.join(root_dir, "src", "worksheets", "config", "credentials.json")
creds2 = os.path.join(root_dir, "src", "worksheets", "config", "service_account.json")
creds3 = os.path.join(root_dir, "src", "worksheets", "config", "token.json")

for cred in [creds1, creds2, creds3]:
  assert os.path.exists(cred), f"Cannot find the credential file: {cred}"

# Step2 HF-Agent Prototyping Loop

In [ ]:
# Config and LLM initialization
import sys, json, os
from worksheets.agent.config import Config, OpenAIModelConfig
from worksheets.llm.prompts import init_llm

PROMPTS_DIR = "/content/genie-worksheets/src/worksheets/prompts"
DOTENV_PATH = "/content/genie-worksheets/.env"

init_llm(PROMPTS_DIR, DOTENV_PATH)

config = Config(
    semantic_parser = OpenAIModelConfig(model_name="gpt-4.1"),
    response_generator = OpenAIModelConfig(model_name="gpt-4.1"),
    knowledge_parser = OpenAIModelConfig(model_name="gpt-4.1"),
    knowledge_base   = OpenAIModelConfig(model_name="gpt-4.1"),
    validate_response=False,
    prompt_log_path = "logs.log",
    conversation_log_path = "conv_log.json",
)



In [ ]:
# Definition of HF-Agent API function
from worksheets.agent.config import agent_api

# Reading KB rules from JSON file
KB_PATH = os.path.join(os.path.dirname(os.getcwd()), "knowledge_base", "rules.json")
if not os.path.exists(KB_PATH):
    # Upload rules.json path below
    KB_PATH = "/content/knowledge_base/rules.json"
with open(KB_PATH, "r") as f:
    KB = json.load(f)


def _eval_flags(ps: dict) -> tuple[list[str], str]:
    v = (ps.get("vitals") or {})
    labs = (ps.get("labs") or {})
    sym = (ps.get("symptoms") or [])
    flags: list[str] = []
    if v.get("sbp", 999) < 90 and any(s in sym for s in ["dizziness","syncope","lightheadedness"]):
        flags.append("symptomatic_hypotension")
    if v.get("sbp", 999) < 80:
        flags.append("severe_hypotension")
    k = labs.get("potassium_mmol_l")
    if k is not None and k > 5.5:
        flags.append("hyperkalemia_moderate")
    if k is not None and k >= 6.0:
        flags.append("hyperkalemia_severe")
    egfr = labs.get("egfr")
    if egfr is not None and egfr < 20:
        flags.append("egfr_critical")
    if v.get("hr", 999) < 50:
        flags.append("bradycardia")
    high = {"symptomatic_hypotension","severe_hypotension","hyperkalemia_severe","egfr_critical"}
    level = "high" if any(f in high for f in flags) else ("moderate" if flags else "none")
    return flags, level


@agent_api("evaluate_risk", "Evaluate red-flags and risk level using KB")
def evaluate_risk(
    vitals_sbp: int | None = None,
    vitals_dbp: int | None = None,
    vitals_hr: int | None = None,
    labs_creatinine_mg_dl: float | None = None,
    labs_egfr: int | None = None,
    labs_potassium_mmol_l: float | None = None,
    symptoms: list[str] | None = None,
) -> dict:
    ps = {
        "vitals": {"sbp": vitals_sbp, "dbp": vitals_dbp, "hr": vitals_hr},
        "labs": {"creatinine_mg_dl": labs_creatinine_mg_dl, "egfr": labs_egfr, "potassium_mmol_l": labs_potassium_mmol_l},
        "symptoms": symptoms or []
    }
    flags, level = _eval_flags(ps)
    return {"risk_level": level, "flags": flags}


@agent_api("create_recommendation", "Create per-visit titration recommendation based on KB")
def create_recommendation(
    vitals_sbp: int | None = None,
    vitals_dbp: int | None = None,
    vitals_hr: int | None = None,
    labs_creatinine_mg_dl: float | None = None,
    labs_egfr: int | None = None,
    labs_potassium_mmol_l: float | None = None,
    meds: list[dict] | None = None,
) -> dict:
    ps = {
        "vitals": {"sbp": vitals_sbp, "dbp": vitals_dbp, "hr": vitals_hr},
        "labs": {"creatinine_mg_dl": labs_creatinine_mg_dl, "egfr": labs_egfr, "potassium_mmol_l": labs_potassium_mmol_l},
        "meds": meds or []
    }
    flags, level = _eval_flags(ps)
    if level == "high":
        return {
            "rec_actions": ["hold related agents per KB"],
            "rec_monitoring": [{"when": "1w", "labs": ["BMP","Cr","eGFR","K+"]}],
            "rec_followup_weeks": 1,
            "rec_tags": ["red_flag"],
        }

    # naive ARNI uptitration example
    entresto = next((m for m in ps["meds"] if "sacubitril/valsartan" in (m.get("name") or "").lower()), None)
    if entresto and "49/51" in (entresto.get("dose") or ""):
        actions = [{"drug": "sacubitril/valsartan", "change": "49/51mg bid → 97/103mg bid"}]
        monitoring = [{"when": "1-2w", "labs": ["BMP","Cr","eGFR","K+"]}]
    else:
        actions = ["maintain current doses"]
        monitoring = [{"when": "as_needed"}]
    return {
        "rec_actions": actions,
        "rec_monitoring": monitoring,
        "rec_followup_weeks": 2,
        "rec_tags": [],
    }



In [64]:
# HF-Agent Construction
from worksheets import AgentBuilder, conversation_loop
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="ERROR")

botname = "HF-Agent"
description = "An assistant that collects patient data, screens red-flags, and proposes safe titration plans per HF guidelines."
starting_prompt = """Hi! I'm your heart failure medication assistant. I can help you safely manage medication titration and monitoring.
                    To get started, please share your name, current HF medications and doses, your latest BP/HR/weight,
                    recent labs (creatinine/eGFR/potassium), and any symptoms or missed doses."""

GSHEET_HF_ID = "1cx6Q0u6lh1Jvlg0jw2zbqo7Tgcq4pkKnMg__d4ytENc"

agent_builder = (
    AgentBuilder(
        name=botname,
        description=description,
        starting_prompt=starting_prompt,
    )
    .with_gsheet_specification(GSHEET_HF_ID)
)

agent = agent_builder.build(config)
print("Worksheets:")
for ws in agent.runtime.genie_worksheets:
    print(ws)



Worksheets:
Intake(patient_name: str, vitals_sbp: int, vitals_dbp: int, vitals_hr: int, vitals_weight_kg: float, labs_creatinine_mg_dl: float, labs_egfr: int, labs_potassium_mmol_l: float, symptoms: str, adherence: str, meds: str)
Communicate()
Escalation()
RiskScreen(confirm_risk: bool)
Recommendation()


In [65]:
with agent:
  await conversation_loop(agent, debug=False)

# print("Ready. Please set GSHEET_HF_ID and then run the conversation loop cell as in RideBooking example.")


Agent: Hi! I'm your heart failure medication assistant. I can help you safely manage medication titration and monitoring. To get started, please share your name, current HF medications and doses, your latest BP/HR/weight, recent labs (creatinine/eGFR/potassium), and any symptoms or missed doses.
User: My name is Andrew

intake = Intake(patient_name='Andrew')
User: My name is Andrew
Agent: Could you please provide Andrew's most recent systolic blood pressure (in mmHg)?
User: Now my BP is around  110/70 mmHg.

intake.vitals_sbp = 110
intake.vitals_dbp = 70
User: My name is Andrew
Agent: Could you please provide Andrew's most recent systolic blood pressure (in mmHg)?
User: Now my BP is around  110/70 mmHg.
Agent: What is Andrew's resting heart rate in beats per minute (bpm)?
User: My BPM is about 65 in resting

intake.vitals_hr = 65
User: My name is Andrew
Agent: Could you please provide Andrew's most recent systolic blood pressure (in mmHg)?
User: Now my BP is around  110/70 mmHg.
Agent:

Traceback (most recent call last):
  File "/content/genie-worksheets/src/worksheets/utils/interface.py", line 89, in conversation_loop
    user_utterance = input_user()
                     ^^^^^^^^^^^^
  File "/content/genie-worksheets/src/worksheets/utils/interface.py", line 46, in input_user
    user_utterance = input(bcolors.OKCYAN + bcolors.BOLD + "User: ")
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 1177, in raw_input
    return self._input_request(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 1230, in _input_request
    raise EOFError
EOFError
